In [ ]:
%matplotlib widget

# Env setup
from dotenv import load_dotenv
load_dotenv()

# CUDA selection
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch version:", torch.__version__)
if torch.cuda.is_available():
    print(f"GPU available. PyTorch expects CUDA v{torch.version.cuda}")
else:
    print("No CUDA detected. Using CPU")

# Model set-up
from transformers import Sam3Processor, Sam3Model
model = Sam3Model.from_pretrained("facebook/sam3").to(device)
processor = Sam3Processor.from_pretrained("facebook/sam3")

In [ ]:
import io

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import PIL.Image
import requests
from IPython.display import clear_output, display, HTML
from matplotlib.patches import Rectangle

class Sam3Objs:
    '''Class to hold datatypes from SAM3 outputs'''

    def __init__(self, model, processor):
        self.model = model                  # SAM3 model instance (for running inference)
        self.processor = processor          # SAM3 processor instance (for processing datatypes)
        self.img = None                     # Image to use for inference
        self.img_size = None
        self.original_sizes = None
        self.img_embed = None               # Stored image embeddings for image
        self.raw_output = None              # Raw outputs before post-processing
        self.results = None                 # Post-processer results containing masks, scores, and bounding boxes

    def clear_results(self):
        '''Clears inference results (raw output, masks, and scores)'''
        self.raw_output = None
        self.results = None

    def is_img_loaded(self):
        return self.img is not None and self.img_embed is not None

    def is_result_ready(self):
        '''If results are present (inference has been run), returns True. If not, returns False.'''
        return self.raw_output is not None and self.results is not None

    def set_img(self, image:PIL.Image):
        '''Sets the current image for inference and stores vision embeddings.'''
        if self.is_result_ready():
            self.clear_results()
        self.img = image
        self.img_size = image.size
        img_inputs = self.processor(images=image, return_tensors="pt").to(self.model.device)
        self.original_size = img_inputs.original_sizes
        with torch.no_grad():
            self.img_embed = self.model.get_vision_features(pixel_values=img_inputs.pixel_values)

        return self.img

    def _construct_prompt(self,
                        text:str = None,
                        boxes:dict = None):
        '''Consolidates prompt into a usable format.
        Either text or bounding boxes should be included for a valid prompt
        
        Argss:
            text: str
            boxes: list with each element as a dict of form 
                   {"box": [x_min, y_min, x_max, y_max], "label": 0 | 1}
        '''
        prompt = {}

        # Check text
        text = text.strip()
        if text: prompt['text'] = text

        # Check boxes
        if boxes and len(boxes) > 0:
            prompt_boxes, prompt_blabels = zip(*[(box['box'], box['label']) for box in boxes if 'box' in box and 'label' in box])
            prompt_boxes = list(prompt_boxes)
            prompt_blabels = list(prompt_blabels)
            box_entries = [len(b)==4 for b in prompt_boxes]
            if False in box_entries: prompt_boxes = None
            label_entries = [p==0 or p==1 for p in prompt_blabels]
            if False in label_entries: prompt_blabels = None
        
            if prompt_boxes and prompt_blabels and len(prompt_boxes)==len(prompt_blabels):
                prompt['input_boxes'] = [prompt_boxes] #nested an extra layer because model enables batching
                prompt['input_boxes_labels'] = [prompt_blabels]

        # Compile
        if prompt:
            return prompt
        else:
            raise ValueError("Invalid prompt. Check that text or bounding boxes were included")
        
    def run_inference(self,
                    text:str = None,
                    boxes:list = None):
        '''Runs inference based on prompt.'''
        if not self.is_img_loaded():
            raise Exception("Image must be loaded before running inference.")
        
        prompt = self._construct_prompt(text, boxes)
        
        inputs = self.processor(**prompt,
                                original_sizes = self.original_size,
                                return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            outputs = self.model(vision_embeds=self.img_embed,
                                **inputs
                            )
        self.raw_output = outputs

    def threshold_results(self,
                          instance_thresh:float = 0.5,
                          mask_thres:float = 0.5):
        '''Processes raw outputs based on desired prediction threshold values for
        instance (objectness) and mask confidence (prompt-matching)
        
        Results are a dictionary containing:
        - masks: Binary masks resized to original image size
        - boxes: Bounding boxes in absolute pixel coordinates (xyxy format)
        - scores: Confidence scores
        '''
        results = self.processor.post_process_instance_segmentation(
                self.raw_output,
                threshold=instance_thresh,
                mask_threshold=mask_thres,
                target_sizes=self.original_size.tolist()
                )[0]
        self.results = results
        return results


class Sam3SegmentationWidget:
    """Interactive Jupyter widget for SAM3 segmentation with text and box prompts."""

    def __init__(self, model, processor):
        """
        Initialize the segmentation widget.

        Args:
            processor: Sam3Processor instance
        """
        self.sam3 = Sam3Objs(model, processor)
        self.current_image_array = None
        self.proposed_boxes = None
        self.box_mode = "positive"
        self.drawing_box = False
        self.box_start = None
        self.current_rect = None

        self._setup_ui()
        self._setup_plot()

    def _setup_ui(self):
        """Set up the UI components."""
        self.upload_widget = widgets.FileUpload(
            accept="image/*", multiple=False, description="Upload Image"
        )
        self.upload_widget.observe(self._on_image_upload, names="value")

        self.url_input = widgets.Text(
            placeholder="Or enter image URL",
        )
        self.url_button = widgets.Button(description="Load URL", button_style="info")
        self.url_button.on_click(self._on_load_url)
        url_box = widgets.HBox(
            [self.url_input, self.url_button],
            layout=widgets.Layout(width="100%", justify_content="space-between"),
        )

        self.text_input = widgets.Text(
            placeholder='Enter segmentation prompt (e.g., "person", "dog")',
            continuous_update=False,
        )

        self.box_mode_buttons = widgets.ToggleButtons(
            options=["Positive Boxes", "Negative Boxes"],
            description="Box Mode:",
            button_style="",
            tooltips=[
                "Draw boxes around objects to include",
                "Draw boxes around objects to exclude",
            ],
        )
        self.box_mode_buttons.observe(self._on_box_mode_change, names="value")

        self.clear_box_button = widgets.Button(
            description="Clear Box Prompts", button_style="warning"
        )
        self.clear_box_button.on_click(self._on_clear_boxes)
        box_widgets = widgets.HBox(
            [self.box_mode_buttons, self.clear_box_button],
            layout=widgets.Layout(width="100%",
                justify_content="space-between",
                align_items='flex-end'),
        )

        self.segment_button = widgets.Button(description="Segment", button_style="success")
        self.segment_button.on_click(self._on_submit_prompt)

        self.instance_slider = widgets.FloatSlider(
            value=0.5,
            min=0.0,
            max=1.0,
            step=0.01,
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.instance_slider.observe(self._on_slider_change, names="value")
        inst_thresh =  widgets.HBox(
            [widgets.Label("Object-ness Confidence:"), self.instance_slider],
            layout=widgets.Layout(width="100%", justify_content="space-between"),
        )

        self.mask_slider = widgets.FloatSlider(
            value=0.5,
            min=0.0,
            max=1.0,
            step=0.01,
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.mask_slider.observe(self._on_slider_change, names="value")
        mask_thresh =  widgets.HBox(
            [widgets.Label("Mask Confidence:"), self.mask_slider],
            layout=widgets.Layout(width="100%", justify_content="space-between"),
        )

        self.size_slider = widgets.IntSlider(
            value=960,
            min=300,
            max=2000,
            step=10,
            description="Image Size:",
            continuous_update=False,
            style={"description_width": "initial"},
        )
        self.size_slider.observe(self._on_size_change, names="value")

        self.output = widgets.Output()
        self.status_label = widgets.Label(value="Upload an image to begin")

        # This box will hold our matplotlib output and we can target it with CSS.
        self.plot_container = widgets.Box([self.output])
        self.plot_container.add_class("no-drag")

        # CSS to make the cursor a crosshair over the matplotlib canvas
        css_style = widgets.HTML(
            """
        <style>
            .jupyter-matplotlib-canvas, canvas {
                cursor: crosshair !important;
            }
        </style>
        """
        )
        # Create VBoxes for each accordion pane
        source_pane = widgets.VBox([self.upload_widget, url_box])
        prompt_pane = widgets.VBox(
            [
                widgets.Label("Text Prompt:"),
                self.text_input,
                box_widgets,
                #self.box_mode_buttons,
                #self.clear_box_button,
                #self.instance_slider,
                inst_thresh,
                mask_thresh,
                #self.mask_slider,
                self.segment_button,
            ]
        )
        display_pane = widgets.VBox([self.size_slider])

        # Create the Accordion to hold the control panes
        self.accordion = widgets.Accordion(
            children=[source_pane, prompt_pane, display_pane]
        )
        self.accordion.set_title(0, "Image Source")
        self.accordion.set_title(1, "Segmentation Prompts")
        self.accordion.set_title(2, "Display Settings")
        self.accordion.selected_index = 0  # Start with the first pane open

        # Create the left sidebar for controls
        sidebar = widgets.VBox(
            [self.status_label, widgets.HTML("<h4>Controls</h4>"), self.accordion]
        )
        sidebar.layout = widgets.Layout(
            width="760px",
            min_width="380px",
            max_width="760px",
            border="1px solid #e0e0e0",
            padding="10px",
            margin="0 15px 0 0",
            flex="0 0 auto",
        )

        # Create the main area for the image display
        main_area = widgets.VBox([self.plot_container])
        main_area.layout = widgets.Layout(flex="1", min_width="500px", overflow="auto")

        # Combine sidebar and main area into the final app layout
        app_layout = widgets.VBox([sidebar, main_area])
        app_layout.layout = widgets.Layout(
            width="100%",
            display="flex",
            flex_flow="row",
            align_items="stretch",
        )

        # Set the main container
        self.container = widgets.VBox(
            [
                css_style,
                widgets.HTML("<h3>🖼️ SAM3 Interactive Segmentation</h3>"),
                app_layout,
            ]
        )

    def _setup_plot(self):
        """Set up the matplotlib figure."""
        # plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(12, 8))
        # plt.ion()
        self.ax.axis("off")
        self.fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        self.fig.canvas.toolbar_visible = False
        self.fig.canvas.header_visible = False
        self.fig.canvas.footer_visible = False
        self.fig.canvas.resizable = False

        # plt.close(self.fig)

    def _set_loading(self, is_loading, message="Processing..."):
        """Show/hide loading state and disable/enable controls."""
        if is_loading:
            self.status_label.value = f"⏳ {message}"
            self.upload_widget.disabled = True
            self.url_button.disabled = True
            self.segment_button.disabled = True
            self.clear_box_button.disabled = True
            self.box_mode_buttons.disabled = True
            self.instance_slider.disabled = True
            self.mask_slider.disabled = True
        else:
            self.upload_widget.disabled = False
            self.url_button.disabled = False
            self.segment_button.disabled = False
            self.clear_box_button.disabled = False
            self.box_mode_buttons.disabled = False
            self.instance_slider.disabled = False
            self.mask_slider.disabled = False

    def _on_image_upload(self, change):
        """Handle image upload."""
        if change["new"]:
            uploaded_file = change["new"][0]
            image = PIL.Image.open(io.BytesIO(uploaded_file["content"])).convert("RGB")
            self._set_image(image)

    def _on_load_url(self, button):
        """Handle loading image from URL."""
        url = self.url_input.value.strip()
        if not url:
            self.status_label.value = "Please enter a URL"
            return

        self._set_loading(True, "Downloading image from URL...")

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()
            image = PIL.Image.open(io.BytesIO(response.content)).convert("RGB")
            self._set_image(image)
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error loading image: {str(e)}"

    def _set_image(self, image):
        """Set the current image, adjust figure size, and initialize state."""
        self._set_loading(True, "Processing image through model...")

        try:
            img = self.sam3.set_img(image)
            self.current_image_array = np.asarray(img)
            self._set_loading(False)
            self.status_label.value = (
                f"Image loaded: {image.size[0]}x{image.size[1]} pixels"
            )
            self._resize_figure()
            self._update_display()
            self._connect_plot_events()
            self.accordion.selected_index = 1
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error processing image: {str(e)}"

    def _on_submit_prompt(self, button):
        """Handle running segmentation w/ prompt inputs"""
        self._set_loading(True, f'Segmenting...')
        self.sam3.clear_results()
        text_prompt = self.text_input.value.strip()

        try:
            self.sam3.run_inference(text=text_prompt,
                                    boxes = self.proposed_boxes
                                    )
            self._set_loading(False)
            self.status_label.value = f'Finished segmentation!'
            self.sam3.threshold_results(mask_thres = self.mask_slider.value,
                                        instance_thresh= self.instance_slider.value)
            self.status_label.value += f'{len(self.sam3.results['masks'])} matching objects found'
            self._update_display()
        except Exception as e:
            self._set_loading(False)
            self.status_label.value = f"Error: {str(e)}"

    def _on_box_mode_change(self, change):
        """Handle box mode toggle."""
        self.box_mode = "positive" if change["new"] == "Positive Boxes" else "negative"

    def _on_clear_boxes(self, button):
        """Clear box prompts and reset to image only."""
        try:
            self._set_loading(True, "Clearing prompts and resetting...")
            self.sam3.clear_results()
            self.proposed_boxes = None
            self._set_loading(False)
            self.status_label.value = "Cleared boxes prompts"
            self._update_display()
        except Exception as e:
            self._set_loading(False)
            import traceback
            self.status_label.value = f"Error: {str(e)} {traceback.format_exc()}"

    def _on_slider_change(self, change):
        """Handle confidence threshold change."""
        if self.sam3.is_result_ready():
            self.sam3.threshold_results(mask_thres = self.mask_slider.value,
                                        instance_thresh= self.instance_slider.value)
            self._update_display()

    def _connect_plot_events(self):
        """Connect matplotlib event handlers for box drawing."""
        # Disable matplotlib's toolbar navigation to allow custom box drawing
        if hasattr(self.fig.canvas, "toolbar") and self.fig.canvas.toolbar is not None:
            self.fig.canvas.toolbar.pan()
            self.fig.canvas.toolbar.pan()

        self.fig.canvas.mpl_connect("button_press_event", self._on_press)
        self.fig.canvas.mpl_connect("button_release_event", self._on_release)
        self.fig.canvas.mpl_connect("motion_notify_event", self._on_motion)

    def _on_press(self, event):
        """Handle mouse press for box drawing."""
        if event.inaxes != self.ax:
            return
        self.drawing_box = True
        self.box_start = (event.xdata, event.ydata)

    def _on_motion(self, event):
        """Handle mouse motion for box preview."""
        if not self.drawing_box or event.inaxes != self.ax or self.box_start is None:
            return

        if self.current_rect is not None:
            self.current_rect.remove()

        x0, y0 = self.box_start
        x1, y1 = event.xdata, event.ydata
        width = x1 - x0
        height = y1 - y0

        color = "green" if self.box_mode == "positive" else "red"
        self.current_rect = Rectangle(
            (x0, y0),
            width,
            height,
            fill=False,
            edgecolor=color,
            linewidth=2,
            linestyle="--",
        )
        self.ax.add_patch(self.current_rect)
        self.fig.canvas.draw_idle()

    def _on_release(self, event):
        """Handle mouse release to finalize box."""
        if not self.drawing_box or event.inaxes != self.ax or self.box_start is None:
            self.drawing_box = False
            return

        self.drawing_box = False

        if self.current_rect is not None:
            self.current_rect.remove()
            self.current_rect = None

        if not self.sam3.is_img_loaded():
            return

        x0, y0 = self.box_start
        x1, y1 = event.xdata, event.ydata

        x_min = min(x0, x1)
        x_max = max(x0, x1)
        y_min = min(y0, y1)
        y_max = max(y0, y1)

        if abs(x_max - x_min) < 5 or abs(y_max - y_min) < 5:
            return

        label = self.box_mode == "positive"
        mode_str = "positive" if label else "negative"

        # Store the prompted box in pixel coordinates for display
        if not self.proposed_boxes:
            self.proposed_boxes = []
        self.proposed_boxes.append(
            {"box": [x_min, y_min, x_max, y_max], "label": label}
        )

        self.status_label.value = f"Added {mode_str} box. Total boxes: {len(self.proposed_boxes)}"
        self._update_display()


    def _resize_figure(self):
        """Calculate and apply new figure size based on image and slider value."""
        if not self.sam3.is_img_loaded():
            return

        # 1. Get original image dimensions
        img_w, img_h = self.sam3.img_size

        # 2. The slider's value is now the direct target width for the display
        display_w = float(self.size_slider.value)

        # 3. Calculate the corresponding height to maintain the original aspect ratio
        aspect_ratio = img_h / img_w
        display_h = int(display_w * aspect_ratio)

        # 4. Convert pixel dimensions to inches for Matplotlib and apply
        dpi = self.fig.dpi
        new_figsize = (display_w / dpi, display_h / dpi)
        self.fig.set_size_inches(new_figsize, forward=True)

    def _on_size_change(self, change):
        """Handle a change from the image size slider."""
        if self.sam3.is_img_loaded():
            self._resize_figure()
            # After resizing the canvas, we must redraw the content
            self._update_display()

    def _update_display(self):
        """Update the display with current results."""
        if not self.sam3.is_img_loaded():
            return

        with self.output:
            clear_output(wait=True)

            self.ax.clear()
            self.ax.axis("off")
            self.ax.imshow(self.current_image_array)

            if self.sam3.is_result_ready():
                masks = self.sam3.results.get("masks", [])
                boxes = self.sam3.results.get("boxes", [])
                scores = self.sam3.results.get("scores", [])

                if len(masks) > 0:
                    mask_overlay = np.zeros((*self.current_image_array.shape[:2], 4))

                    for i, (mask, box, score) in enumerate(zip(masks, boxes, scores)):
                        mask_np = mask.cpu().numpy()

                        color = plt.cm.tab10(i % 10)[:3]
                        mask_overlay[mask_np > 0.5] = (*color, 0.5)

                        x0, y0, x1, y1 = box.cpu().numpy()
                        rect = Rectangle(
                            (x0, y0),
                            x1 - x0,
                            y1 - y0,
                            fill=False,
                            edgecolor=color,
                            linewidth=2,
                        )
                        self.ax.add_patch(rect)

                        self.ax.text(
                            x0,
                            y0 - 5,
                            f"{score:.2f}",
                            color="white",
                            fontsize=10,
                            bbox=dict(
                                facecolor=color, alpha=0.7, edgecolor="none", pad=2
                            ),
                        )

                    self.ax.imshow(mask_overlay)
                    self.status_label.value = f"Found {len(masks)} object(s)"
                else:
                    self.status_label.value = (
                        "No objects found above confidence threshold"
                    )

            # Display prompted boxes with dashed lines
            if self.proposed_boxes:
                for prompted_box in self.proposed_boxes:
                    box_coords = prompted_box["box"]
                    is_positive = prompted_box["label"]

                    x0, y0, x1, y1 = box_coords
                    color = "green" if is_positive else "red"

                    rect = Rectangle(
                        (x0, y0),
                        x1 - x0,
                        y1 - y0,
                        fill=False,
                        edgecolor=color,
                        linewidth=2,
                        linestyle="--",
                    )
                    self.ax.add_patch(rect)

            # display(self.fig.canvas)

    def display(self):
        display(self.container)

    # Add this for more convenient display in notebooks
    def _ipython_display_(self):
        self.display()


In [ ]:
widget = Sam3SegmentationWidget(model, processor)
widget.display()